# Smart Cities Traffic Pipeline - 8 Fases con Spark

## Fases del Pipeline:
1. **FASE 1**: Spark lee y procesa tus datos CSV existentes
2. **FASE 2**: Spark Streaming simula ingesta en vivo
3. **FASE 3**: Arquitectura Lambda (batch + streaming juntos)
4. **FASE 4**: MLlib clasifica niveles de congestión
5. **FASE 5**: MLlib predice incidentes
6. **FASE 6**: Alertas automáticas
7. **FASE 7**: Recomendaciones de rutas
8. **FASE 8**: Dashboard Grafana completo

### Datos disponibles:
- `data/raw/`: CSVs históricos (2025-08 a 2026-01)
- Columnas: `idTram`, `data` (timestamp), `estatActual`, `estatPrevist`
- Estados: 0 = Normal, 1 = Moderado, 2 = Alto

In [ ]:
#  IMPORTANTE: Si sigues viendo el mismo error, reinicia el kernel
# Presiona: Kernel → Restart Kernel (o Ctrl+Shift+R)
# Luego ejecuta esta celda de nuevo

import sys
import os
sys.path.insert(0, '../scripts')

# Recargar módulo si ya fue importado
if 'spark_pipeline' in sys.modules:
    del sys.modules['spark_pipeline']

from spark_pipeline import SparkTrafficPipeline
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✓ Módulo recargar correctamente")
print("✓ Librerías importadas")

✓ Módulo recargar correctamente
✓ Librerías importadas


# FASE 1: Lectura y Procesamiento Batch de CSVs

Lee archivos CSV históricos y realiza transformaciones iniciales.
- Parseado de timestamps
- Limpieza de datos
- Estadísticas descriptivas

In [ ]:
# Fase 1: Inicializar pipeline Spark
pipeline = SparkTrafficPipeline(app_name="SmartCities_Pipeline") #Instaciar objeto de la clase
spark = pipeline.spark                                           #Definir atributo-propiedad .spark -> Sesion

# Listar archivos CSV disponibles
import glob
csv_files = glob.glob("../data/raw/*.csv")
csv_files = [f for f in csv_files if "Tramos" not in f]  # Excluir archivos de configuración
csv_files.sort() #Ordena 

print(f"📁 CSVs encontrados ({len(csv_files)}):")
for f in csv_files[:5]:
    print(f"  - {os.path.basename(f)}")

26/04/21 11:19:22 WARN Utils: Your hostname, isaac-ThinkPad-T14-Gen-2i resolves to a loopback address: 127.0.1.1; using 192.168.18.88 instead (on interface wlp0s20f3)
26/04/21 11:19:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 11:19:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
INFO:spark_pipeline:✓ SparkSession creada: SmartCities_Pipeline


📁 CSVs encontrados (6):
  - 2025-08.csv
  - 2025-09.csv
  - 2025-10.csv
  - 2025-11.csv
  - 2025-12.csv


In [ ]:
# Fase 1: Cargar y combinar CSVs
df_raw = pipeline.load_multiple_csv(csv_files) # Carga y combinación de CSVs en un DataFrame de Spark

# Preprocesar datos
df_processed = pipeline.preprocess_traffic_data(df_raw) # Limpieza, transformación  de datos

# Resumen
pipeline.show_summary(df_processed, "FASE 1: Datos Procesados", rows=15)

# Estadísticas por tramo
stats_df = pipeline.compute_statistics(df_processed)
print("\n📊 Estadísticas por Tramo (Top 10):")
print(stats_df.head(10).to_string())

INFO:spark_pipeline:✓ Datos cargados: 1740704 filas, esquema: 4 columnas        
INFO:spark_pipeline:  ✓ 2025-08.csv - esquema estándar (4 cols)
INFO:spark_pipeline:✓ Datos cargados: 1927968 filas, esquema: 4 columnas        
INFO:spark_pipeline:  ✓ 2025-09.csv - esquema estándar (4 cols)
INFO:spark_pipeline:✓ Datos cargados: 1997660 filas, esquema: 4 columnas        
INFO:spark_pipeline:  ✓ 2025-10.csv - esquema estándar (4 cols)
INFO:spark_pipeline:✓ Datos cargados: 2030644 filas, esquema: 4 columnas        
INFO:spark_pipeline:  ✓ 2025-11.csv - esquema estándar (4 cols)
INFO:spark_pipeline:✓ Datos cargados: 2266320 filas, esquema: 4 columnas        
INFO:spark_pipeline:  ✓ 2025-12.csv - esquema estándar (4 cols)
INFO:spark_pipeline:✓ Datos cargados: 311766 filas, esquema: 8 columnas         
INFO:spark_pipeline:  ✓ 2026_01.csv - esquema alternativo (8 cols), mapeado
INFO:spark_pipeline:✓ CSVs combinados: 10275062 filas totales                   
INFO:spark_pipeline:✓ Datos preproces


  FASE 1: Datos Procesados


Total de registros: 9410268
Columnas: 6
+------+--------------+-----------+------------+-------------------+----+
|idTram|data          |estatActual|estatPrevist|timestamp          |hour|
+------+--------------+-----------+------------+-------------------+----+
|1     |20250801001053|1          |1           |2025-08-01 00:10:53|0   |
|2     |20250801001053|0          |0           |2025-08-01 00:10:53|0   |
|3     |20250801001053|0          |0           |2025-08-01 00:10:53|0   |
|4     |20250801001053|2          |2           |2025-08-01 00:10:53|0   |
|5     |20250801001053|1          |1           |2025-08-01 00:10:53|0   |
|6     |20250801001053|0          |0           |2025-08-01 00:10:53|0   |
|7     |20250801001053|0          |0           |2025-08-01 00:10:53|0   |
|8     |20250801001053|1          |1           |2025-08-01 00:10:53|0   |
|9     |20250801001053|0          |0           |2025-08-01 00:10:53|0   |
|10    |20250801001053|1          |1           |2025-08-01 00:10:53|0   

INFO:spark_pipeline:✓ Estadísticas calculadas                                   



📊 Estadísticas por Tramo (Top 10):
   idTram  avg_congestion  std_congestion  max_congestion  min_congestion  record_count
0      12        0.572415        0.726759               2               0         21715
1      13        0.560536        0.549919               2               0         22689
2      14        0.615013        0.712642               2               0         20689
3      18        0.685519        0.884233               2               0         20793
4      38        0.475112        0.741269               2               0         22641
5      46        0.000000        0.000000               0               0         22725
6      67        0.503988        0.768553               2               0         22691
7      70        1.032690        0.792376               2               0         19027
8      93        0.867432        0.811274               2               0         15245
9     107        1.141264        0.854931               2               0         17

# FASE 2: Spark Streaming - Simulación de Ingesta en Vivo

Simula ingesta de datos en tiempo real usando RDD streaming.
Sin Kafka (demasiado complejo para este ambiente), usamos simulación con minibatches.

In [4]:
# Fase 2: Simular ingesta de datos en tiempo real
print("🔄 FASE 2: Iniciando simulación de streaming...\n")

# Crear generador de batches
streaming_generator = pipeline.simulate_streaming(df_processed, batch_interval_ms=1000)

# Procesar algunos batches de ejemplo (primeros 5)
streaming_batches = []
print("📊 Procesando 5 minibatches de streaming simulado:\n")

for i, batch in enumerate(streaming_generator):
    if i >= 5:
        break
    
    # Procesar batch
    batch_processed = pipeline.process_streaming_batch(batch)
    streaming_batches.append(batch_processed)
    
    print(f"  Batch {i+1}: {batch_processed.count()} registros")

# Combinar todos los batches
df_streaming_combined = streaming_batches[0]
for batch in streaming_batches[1:]:
    df_streaming_combined = df_streaming_combined.union(batch)

print(f"\n✓ Total registros streaming: {df_streaming_combined.count()}")
print(f"✓ Tiempo de procesamiento: Simulado en minibatches")

🔄 FASE 2: Iniciando simulación de streaming...

📊 Procesando 5 minibatches de streaming simulado:



INFO:spark_pipeline:✓ Streaming simulado: 94319 registros en batches            


  Batch 1: 18863 registros
  Batch 2: 18863 registros
  Batch 3: 18863 registros
  Batch 4: 18863 registros
  Batch 5: 18863 registros



✓ Total registros streaming: 94315
✓ Tiempo de procesamiento: Simulado en minibatches


# FASE 3: Arquitectura Lambda

Combina:
- **Batch Layer**: Histórico histórico (procesamiento offline)
- **Speed Layer**: Análisis en tiempo real (últimas 3600s)
- **Serving Layer**: Vista unificada para consultas

In [6]:
# Fase 3: Implementar Arquitectura Lambda
print("⚙️ FASE 3: Arquitectura Lambda\n")

# Batch Layer: Histórico completo
batch_layer = pipeline.lambda_batch_layer(df_processed)
print(f"Batch Layer: {batch_layer.count()} registros (histórico)")

# Speed Layer: Análisis en tiempo real sobre streaming
speed_layer = pipeline.lambda_speed_layer(df_streaming_combined)
print(f"Speed Layer: {speed_layer.count()} registros (real-time)")

# Serving Layer: Merge de capas
serving_layer = pipeline.lambda_merge(batch_layer, speed_layer)

print(f"\n✓ Serving Layer creada: {serving_layer.count()} registros")
print("\n📊 Muestra del Serving Layer:")
serving_layer.select(
    "idTram", "timestamp", "estatActual", 
    "rolling_avg", "real_time_alert"
).show(10, truncate=False)

⚙️ FASE 3: Arquitectura Lambda



Batch Layer: 9410268 registros (histórico)


INFO:spark_pipeline:✓ Arquitectura Lambda procesada                             


Speed Layer: 94315 registros (real-time)



✓ Serving Layer creada: 9410268 registros

📊 Muestra del Serving Layer:


+------+-------------------+-----------+-----------+---------------+
|idTram|timestamp          |estatActual|rolling_avg|real_time_alert|
+------+-------------------+-----------+-----------+---------------+
|4     |2025-08-01 00:10:53|2          |NULL       |NULL           |
|11    |2025-08-01 00:10:53|0          |NULL       |NULL           |
|6     |2025-08-01 00:10:53|0          |NULL       |NULL           |
|8     |2025-08-01 00:10:53|1          |NULL       |NULL           |
|10    |2025-08-01 00:10:53|1          |NULL       |NULL           |
|1     |2025-08-01 00:10:53|1          |NULL       |NULL           |
|3     |2025-08-01 00:10:53|0          |NULL       |NULL           |
|7     |2025-08-01 00:10:53|0          |NULL       |NULL           |
|9     |2025-08-01 00:10:53|0          |NULL       |NULL           |
|2     |2025-08-01 00:10:53|0          |NULL       |NULL           |
+------+-------------------+-----------+-----------+---------------+
only showing top 10 rows



# FASE 4: Clasificación de Niveles de Congestión con MLlib

Random Forest Classification:
- **Input**: Hora, ID de tramo
- **Output**: Predicción del nivel de congestión (0=Normal, 1=Moderado, 2=Alto)

In [ ]:
# Fase 4: Entrenar modelo de clasificación
print(" FASE 4: Clasificación de Congestión con MLlib\n")

# Entrenar modelo
classification_model, accuracy, predictions_class = pipeline.train_classification_model(
    df_processed, 
    split_ratio=0.8
)

print(f"📈 Métricas del modelo:")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Mostrar predicciones
print("\n📊 Predicciones (muestra):")
predictions_class.select(
    "idTram", "hour", "estatActual", "prediction", 
).show(10, truncate=False)

# Matriz de confusión
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
print(f"\n✓ F1-Score: {evaluator.setMetricName('f1').evaluate(predictions_class):.4f}")
print(f"✓ Precision: {evaluator.setMetricName('weightedPrecision').evaluate(predictions_class):.4f}")
print(f"✓ Recall: {evaluator.setMetricName('weightedRecall').evaluate(predictions_class):.4f}")

🤖 FASE 4: Clasificación de Congestión con MLlib



INFO:spark_pipeline:✓ Pipeline de clasificación creado
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_6 in memory! (computed 27.0 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_2 in memory! (computed 42.8 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_5 in memory! (computed 27.0 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_3 in memory! (computed 42.8 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_0 in memory! (computed 27.0 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_1 in memory! (computed 42.8 MiB so far)
26/04/21 14:05:28 WARN MemoryStore: Not enough space to cache rdd_615_4 in memory! (computed 42.8 MiB so far)
26/04/21 14:05:28 WARN BlockManager: Persisting block rdd_615_0 to disk instead.
26/04/21 14:05:28 WARN BlockManager: Persisting block rdd_615_1 to disk instead.
26/04/21 14:0

📈 Métricas del modelo:
   Accuracy: 0.6059 (60.59%)

📊 Predicciones (muestra):


+------+----+-----------+----------+
|idTram|hour|estatActual|prediction|
+------+----+-----------+----------+
|1     |0   |1          |0.0       |
|1     |1   |0          |0.0       |
|1     |1   |0          |0.0       |
|1     |2   |0          |0.0       |
|1     |4   |0          |0.0       |
|1     |4   |0          |0.0       |
|1     |5   |1          |0.0       |
|1     |7   |2          |2.0       |
|1     |8   |2          |1.0       |
|1     |8   |2          |1.0       |
+------+----+-----------+----------+
only showing top 10 rows




✓ F1-Score: 0.5715


✓ Precision: 0.5931


✓ Recall: 0.6059


# FASE 5: Predicción de Incidentes de Tráfico

GBT (Gradient Boosted Trees Regression):
- **Detecta**: Cambios bruscos en congestión (0→2)
- **Predice**: Probabilidad de incidente próximo

In [ ]:
# Fase 5: Entrenar modelo de predicción de incidentes
print("⚠️ FASE 5: Predicción de Incidentes de Tráfico\n")

# Entrenar modelo
incident_model, predictions_incident = pipeline.train_incident_model(
    df_processed, 
    split_ratio=0.8
)

print("✓ Modelo de predicción de incidentes entrenado")

# Análisis de incidentes
incidents_detected = predictions_incident.filter(
    (predictions_incident.is_incident == 1) | 
    (predictions_incident.prediction > 0.5)
).select(
    "idTram", "hour", "is_incident", "prediction"
)

print(f"\n📊 Incidentes potenciales detectados: {incidents_detected.count()}")
print("\nTop incidentes:")
incidents_detected.orderBy("prediction", ascending=False).show(10, truncate=False)

# Distribución
print("\n📈 Distribución de predicciones:")
from pyspark.sql.functions import col as spark_col
predictions_incident.select("prediction").summary("count", "mean", "stddev", "min", "max").show()

INFO:spark_pipeline:✓ Pipeline de predicción de incidentes creado


⚠️ FASE 5: Predicción de Incidentes de Tráfico



26/04/21 14:33:51 WARN MemoryStore: Not enough space to cache rdd_856_6 in memory! (computed 29.4 MiB so far)
26/04/21 14:33:51 WARN BlockManager: Persisting block rdd_856_6 to disk instead.
26/04/21 14:33:51 WARN MemoryStore: Not enough space to cache rdd_856_1 in memory! (computed 19.6 MiB so far)
26/04/21 14:33:51 WARN BlockManager: Persisting block rdd_856_1 to disk instead.
26/04/21 14:33:52 WARN MemoryStore: Not enough space to cache rdd_856_4 in memory! (computed 44.0 MiB so far)
26/04/21 14:33:52 WARN BlockManager: Persisting block rdd_856_4 to disk instead.
26/04/21 14:33:52 WARN MemoryStore: Not enough space to cache rdd_856_5 in memory! (computed 19.6 MiB so far)
26/04/21 14:33:52 WARN BlockManager: Persisting block rdd_856_5 to disk instead.
26/04/21 14:33:52 WARN MemoryStore: Not enough space to cache rdd_856_7 in memory! (computed 44.0 MiB so far)
26/04/21 14:33:52 WARN BlockManager: Persisting block rdd_856_7 to disk instead.
26/04/21 14:33:53 WARN MemoryStore: Not enoug

# FASE 6: Sistema de Alertas Automáticas

Genera alertas en tiempo real basadas en:
- Congestión > umbral (1.5)
- Predicción de incidente
- Patrones anómalos

In [ ]:
# Fase 6: Generar alertas automáticas
print("🚨 FASE 6: Sistema de Alertas Automáticas\n")

# Generar alertas
alerts_df = pipeline.generate_alerts(df_processed, threshold_congestion=1.5)

print(f"🔔 Total de alertas generadas: {alerts_df.count()}\n")

# Estadísticas de alertas
alert_stats = pipeline.alert_statistics(alerts_df)
print("📊 Distribución por severidad:")
print(alert_stats.to_string(index=False))

# Muestra de alertas
print("\n⚠️ Últimas alertas:")
alerts_df.orderBy("timestamp", ascending=False).show(15, truncate=False)

# Crear tabla de alertas agrupadas
from pyspark.sql.functions import count as spark_count
alerts_by_tram = alerts_df.groupBy("idTram", "severity").agg(
    spark_count("*").alias("alert_count")
).orderBy("alert_count", ascending=False)

print("\n🎯 Tramos más alertados:")
alerts_by_tram.show(10, truncate=False)

# FASE 7: Motor de Recomendación de Rutas

Recomienda rutas alternativas basadas en congestión en tiempo real:
- **OPTIMA**: Ruta con menor congestión
- **ALTERNATIVA_1**: Segunda mejor opción
- **ALTERNATIVA_2**: Tercera opción

In [ ]:
# Fase 7: Generar recomendaciones de rutas
print("🗺️ FASE 7: Motor de Recomendación de Rutas\n")

# Generar recomendaciones
recommendations_df = pipeline.generate_route_recommendations(df_processed)

print(f"✓ Recomendaciones generadas: {recommendations_df.count()}\n")

# Distribución de recomendaciones
rec_dist = recommendations_df.groupBy("recommendation").agg(
    spark_count("*").alias("count")
).orderBy("count", ascending=False)

print("📊 Distribución de recomendaciones:")
rec_dist.show(truncate=False)

# Recomendaciones por hora
print("\n⏰ Recomendaciones por hora:")
recommendations_df.select("hour", "recommendation").distinct().orderBy("hour").show(25, truncate=False)

# Análisis por tramo
print("\n🎯 Rutas recomendadas (muestra por tramo):")
recommendations_df.select("idTram", "hour", "recommendation").distinct().orderBy("idTram", "hour").show(15, truncate=False)

# Guardar recomendaciones
recommendations_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "../data/processed/route_recommendations"
)
print("\n✓ Recomendaciones guardadas en data/processed/route_recommendations")

# FASE 8: Dashboard Grafana Completo

Exporta métricas a PostgreSQL/TimescaleDB para visualización en Grafana:
- Paneles de congestión en tiempo real
- Historial de alertas
- Predicciones de incidentes
- Recomendaciones de rutas

In [ ]:
# Fase 8: Preparar datos para Grafana
print("📊 FASE 8: Dashboard Grafana\n")

# Preparar tabla de métricas consolidadas
from pyspark.sql.functions import max as spark_max, min as spark_min, avg as spark_avg

metrics_for_grafana = df_processed.groupBy("idTram", "hour").agg(
    spark_avg("estatActual").alias("avg_congestion"),
    spark_max("estatActual").alias("max_congestion"),
    spark_min("estatActual").alias("min_congestion"),
    spark_count("*").alias("observation_count")
).select(
    "idTram", "hour", "avg_congestion", "max_congestion", "min_congestion", "observation_count"
)

print(f"✓ Tabla de métricas preparada: {metrics_for_grafana.count()} registros\n")

# Mostrar muestra
print("📈 Métricas para Grafana:")
metrics_for_grafana.orderBy("max_congestion", ascending=False).show(15, truncate=False)

# Guardar en formato Parquet y CSV para Grafana
print("\n💾 Guardando datos...\n")

# Guardar métricas
pipeline.save_to_parquet(metrics_for_grafana, "../data/processed/metrics_grafana")
metrics_for_grafana.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "../data/processed/metrics_csv"
)

# Guardar alertas
alerts_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "../data/processed/alerts_csv"
)

# Guardar predicciones
predictions_with_type = predictions_class.select(
    "idTram", "hour", "estatActual", "prediction"
).coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "../data/processed/predictions_csv"
)

print("✓ Métricas guardadas → data/processed/metrics_grafana")
print("✓ Alertas guardadas → data/processed/alerts_csv")
print("✓ Predicciones guardadas → data/processed/predictions_csv")

In [ ]:
# Conexión a PostgreSQL/TimescaleDB para Grafana
print("\n🗄️ Conexión a PostgreSQL/TimescaleDB\n")

# Configuración de conexión (ajustar según tu docker-compose)
db_config = {
    "host": "localhost",
    "port": 5432,
    "database": "smart_cities",
    "user": "postgres",
    "password": "postgres"  # Cambiar con tu contraseña real
}

print("⚙️ Configuración de conexión:")
print(f"  Host: {db_config['host']}")
print(f"  Port: {db_config['port']}")
print(f"  Database: {db_config['database']}")

# Intentar conexión
try:
    import psycopg2
    from psycopg2 import sql
    
    # Conectar
    conn = psycopg2.connect(
        host=db_config['host'],
        port=db_config['port'],
        database=db_config['database'],
        user=db_config['user'],
        password=db_config['password']
    )
    cur = conn.cursor()
    
    print("\n✓ ¡Conexión exitosa a PostgreSQL!")
    
    # Crear tabla de métricas si no existe
    cur.execute("""
        CREATE TABLE IF NOT EXISTS traffic_metrics (
            idTram INTEGER,
            hour INTEGER,
            avg_congestion FLOAT,
            max_congestion FLOAT,
            min_congestion FLOAT,
            observation_count INTEGER,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    """)
    conn.commit()
    
    print("✓ Tabla 'traffic_metrics' lista")
    
    # Insertar datos (primeros 1000 registros para demo)
    metrics_pd = metrics_for_grafana.limit(1000).toPandas()
    for idx, row in metrics_pd.iterrows():
        cur.execute("""
            INSERT INTO traffic_metrics 
            (idTram, hour, avg_congestion, max_congestion, min_congestion, observation_count)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            int(row['idTram']), 
            int(row['hour']), 
            float(row['avg_congestion']),
            float(row['max_congestion']),
            float(row['min_congestion']),
            int(row['observation_count'])
        ))
    conn.commit()
    print(f"✓ {len(metrics_pd)} registros insertados en PostgreSQL")
    
    cur.close()
    conn.close()
    
except Exception as e:
    print(f"\n⚠️ No se pudo conectar a PostgreSQL: {str(e)}")
    print("   (Aunque esto es OK - los datos están guardados en CSV/Parquet)")
    print("   Para usar Grafana, asegúrate que Docker esté ejecutándose:")

In [ ]:
print("   cd sumo_db_project && docker-compose up -d")

## 📊 Cómo Usar Grafana

### 1. Iniciar Docker
```bash
cd sumo_db_project
docker-compose up -d
```

### 2. Acceder a Grafana
- **URL**: http://localhost:3000
- **Usuario**: admin
- **Contraseña**: admin

### 3. Configurar Data Source PostgreSQL
1. Settings → Data Sources → Add data source
2. Seleccionar PostgreSQL
3. Configurar:
   - Host: localhost:5432
   - Database: smart_cities
   - User: postgres
   - Password: postgres

### 4. Crear Paneles en Grafana
Ejemplo de Query:
```sql
SELECT timestamp, avg_congestion FROM traffic_metrics 
WHERE idTram = 1 AND timestamp > now() - interval '24 hours'
ORDER BY timestamp
```

### 5. Dashboards Sugeridos
- **Panel 1**: Congestión por hora (línea)
- **Panel 2**: Alertas críticas (tabla)
- **Panel 3**: Predicciones de incidentes (gauge)
- **Panel 4**: Tramos más congestionados (top N)

# 📋 Resumen del Pipeline Completo

In [ ]:
# Resumen final del pipeline
print("="*70)
print(" 🏁 SMART CITIES TRAFFIC PIPELINE - RESUMEN FINAL")
print("="*70)

summary_data = {
    "FASE": [
        "1️⃣ Batch Processing",
        "2️⃣ Streaming Simulado",
        "3️⃣ Lambda Architecture",
        "4️⃣ MLlib Clasificación",
        "5️⃣ Predicción Incidentes",
        "6️⃣ Alertas Automáticas",
        "7️⃣ Recomendaciones Rutas",
        "8️⃣ Grafana"
    ],
    "Estado": [
        f"✓ {df_processed.count()} registros",
        f"✓ {df_streaming_combined.count()} streaming",
        f"✓ Merge completado",
        f"✓ Accuracy: {accuracy:.2%}",
        f"✓ RMSE: Training OK",
        f"✓ {alerts_df.count()} alertas",
        f"✓ {recommendations_df.count()} rutas",
        "✓ Datos exportados"
    ],
    "Output": [
        "data/processed/",
        "Minibatches - 5 procesados",
        "Serving Layer",
        "predictions_csv/",
        "predictions_csv/",
        "alerts_csv/",
        "route_recommendations/",
        "metrics_grafana/"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*70)
print("📁 Archivos generados:")
print("="*70)
print("""
✓ scripts/spark_pipeline.py          - Módulo principal con 8 fases
✓ notebooks/03_smart_cities_pipeline.ipynb  - Este notebook
✓ data/processed/metrics_grafana/    - Datos para Grafana (Parquet)
✓ data/processed/metrics_csv/        - Métricas en CSV
✓ data/processed/alerts_csv/         - Alertas en CSV
✓ data/processed/predictions_csv/    - Predicciones en CSV
✓ data/processed/route_recommendations/ - Rutas en CSV
""")

print("="*70)
print("🚀 Próximos pasos:")
print("="*70)
print("""
1. Iniciar Docker:
   cd sumo_db_project && docker-compose up -d

2. Acceder a Grafana:
   http://localhost:3000 (admin/admin)

3. Configurar PostgreSQL como Data Source en Grafana

4. Crear dashboards con los datos exportados

5. (Opcional) Ejecutar pipeline completo regularmente:
   python scripts/spark_pipeline.py
""")

print("="*70)

# Limpiar Spark
# pipeline.stop()
print("\n✓ ¡Pipeline completado exitosamente!")
print("✓ SparkSession aún activa para exploración interactiva")